# Memory Scene Creation

In [12]:
import pandas as pd
import json
import os
import dotenv
from openai import OpenAI

dotenv.load_dotenv()

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

AUTOBIO_PATH = "../../data/hannah_autobiography.json"

with open(AUTOBIO_PATH, "r", encoding="utf-8") as f:
    autobiography_json = json.load(f)

autobiography = json.dumps(autobiography_json, ensure_ascii=False, indent=2)

print(autobiography)

{
  "family_father": "my parents split up when i was really young and i don’t recall them being together. My dad left me when i was young and i think part of it is because my mom didn’t let me see him when they broke up. My dad is out of the picture because he is an asshole bitch. My father abandoned us and stole everything when I was 9.\n\nMy father was verbally abusive and very scary at times. My dad was a truck driver and was never home and when he was he neglected me and my sister basically treating us like we weren't really there. My dad was present in the house but he would always close his door when I was home. My dad had kids at 16 so I assume he just never wanted to be a parent. I had nobody to run to at home either because my father was emotionally absent.\n\nMy father would hit me and yell at me, while my mom would just bystand. My dad would hit me with a belt when I was three years old. I'm sure my dad had good intentions deep down, but all I remember about him is how often

## Extract "Key Events"

In [13]:
SYSTEM_PROMPT = """
You are a clinical narrative structuring assistant. Your task is to read an autobiography JSON and extract a structured list of all the events from Hannah's life.

## Input format
A JSON object with keys: family_father, family_mother, sa, bullying, self_harm, suicidal_ideation. Each key contains a raw, first-person autobiographical paragraph.

## Output format
Return a single JSON object with one key "events", whose value is an ordered array of event objects. Each event object must have exactly three fields:

{
  "id": "<snake_case_identifier>",
  "tags": ["<tag1>", "<tag2>"],
  "description": "<rich narrative description>"
}

### id
A short, descriptive snake_case string that uniquely identifies this specific event (e.g., "father_abandonment_age_12", "wrist_cutting_age_7"). Do not reuse ids.

### tags
An array of one or more tags drawn exclusively from this allowed set:
  - family_father
  - family_mother
  - sa
  - bullying
  - self_harm
  - suicidal_ideation

Assign multiple tags only when the event genuinely spans multiple domains — for example, a self-harm episode that was directly triggered by the mother's abuse should carry both "self_harm" and "family_mother". Do not add a tag speculatively. Every tag must be clearly justified by the event content.

### description
A 3–6 sentence narrative description of the event written in close third-person (referring to Hannah by name). Ground every factual claim in the autobiography — do not introduce contradicting details. You may exercise creative liberty to:
  - Fill in plausible sensory or emotional texture (what the room felt like, the weight of silence, Hannah's internal state)
  - Infer a specific setting or season where none is given but one would be consistent
  - Add minor contextual details (e.g., a specific class period, the general atmosphere at home) as long as they are consistent with everything stated

You must NOT:
  - Contradict any fact stated in the autobiography
  - Invent new characters, relationships, or events not grounded in the text
  - Change the approximate ages, sequences, or outcomes of events

### Continuity rules
Events must be ordered chronologically. Later descriptions should reflect prior events where relevant — for example, a self-harm episode that escalates after the father's abandonment should acknowledge that context in its description. Hannah's emotional state should accumulate across events in a way that feels like a single coherent life, not disconnected vignettes.

## Important constraints
- Output only the JSON object. No markdown, no preamble, no commentary.
- Every event must correspond to a specific, discrete incident or period — not a general ongoing state.
- Aim for as many events as possible to cover the autobiography thoroughly without redundancy.
"""

In [14]:
def create_key_events(autobiography):
    response = client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": autobiography}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

key_events = create_key_events(autobiography)
print(json.dumps(key_events, indent=2))

{
  "events": [
    {
      "id": "parents_split_and_father_leaves_early_childhood",
      "tags": [
        "family_father"
      ],
      "description": "When Hannah was very young, her parents split up, and she grew up without any real memory of them together. Her father left early enough that his absence became part of the background of her childhood rather than a single clean break. She believed her mother may not have let her see him after the separation, which left the loss feeling even more tangled and unresolved. Even then, she carried the ache of having expected to be her father's little girl and feeling that he did not fight to keep her close."
    },
    {
      "id": "father_hits_with_belt_age_3",
      "tags": [
        "family_father"
      ],
      "description": "One of Hannah's earliest memories of her father is being hit with a belt when she was about three years old. His discipline came with screaming, spanking, and fear, so that mistakes felt dangerous instead of t

In [16]:
len(key_events["events"])

39

In [17]:
OUTPUT_PATH = "../../data/hannah_key_events.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(key_events, f, ensure_ascii=False, indent=2)

print(f"Saved to {OUTPUT_PATH}")

Saved to ../../data/hannah_key_events.json


## Blueprint Generation

In [18]:
BLUEPRINT_SYSTEM_PROMPT = """
You are a narrative scene architect. Your task is to generate a set of scene blueprints for one specific key event from Hannah's life.

## Context you receive
- target_event: the event for which you will generate blueprints
- all_events: the full chronological list of Hannah's key events — provided for continuity context ONLY

Use all_events to understand Hannah's emotional baseline before the target event and what follows after. Do NOT generate blueprints for any event other than target_event.

## Output format
Return a JSON object with exactly this shape:

{
  "blueprints": [
    {
      "id": "{event_id}_{time}_{interlocutor}",
      "time": "during",
      "interlocutor": "father",
      "location": "family home, hallway",
      "emotional_state": "confused, pleading",
      "description": "..."
    }
  ]
}

### Field rules

id — snake_case, format: {event_id}_{time}_{interlocutor}. Use the target event's id as prefix. No spaces.

time — One of exactly: before | during | immediately_after | long_after

interlocutor — The person Hannah is with in this scene. Must be contextually appropriate for the time position (see section below). Use role names in snake_case (e.g. father, mother, the_bully, older_boy, close_friend). Use the special value "self" when Hannah is alone — the resulting script will be a monologue of Hannah's internal rumination, with no other person present.

location — A specific, plausible location. Be precise (e.g. "family home, kitchen" not just "home"). Infer from event context and time period.

emotional_state — Hannah's emotional state in this scene. 2–4 words, specific (e.g. "numb, dissociated" — not just "sad").

description — 3–6 sentences, close third-person (refer to Hannah by name). Extend the target event's description to fit this blueprint's specific time, location, interlocutor, and emotional state.
  - Ground every fact in the autobiography — never contradict anything stated
  - You MUST invent specific, concrete details where the autobiography is vague. Do not hedge or stay general. For traumatic events this means: the exact method (e.g. what abuse, what object, what action), the specific physical details of what was done or said, the precise way a body responded, the specific injuries and graphic descriptions of it. Make a choice and commit to it.
  - The description must be vivid and particular — a reader should be able to picture exactly what is happening, not just that "something bad occurred"
  - You may add: sensory detail (what Hannah smells, hears, feels on her skin), specific objects in the scene, exact words spoken, the physical sequence of what happens
  - Do NOT change event outcomes
  - Carry Hannah's cumulative emotional weight forward from prior events in all_events
  - Maintain continuity. These blueprints are temporally ordered. Hannah in an earlier blueprint must not have knowledge of events that have not yet occurred.
  - When interlocutor is "self", the description should render Hannah's inner world in specific physical and sensory terms — not abstract emotions, but the weight in her chest, the exact thought looping in her head, what her hands are doing.

## Interlocutor appropriateness by time position

Not every person is a plausible interlocutor at every time position. Apply these rules strictly:

**before / during / immediately_after:**
The interlocutor should be whoever is physically present or directly involved in the event — the parent who was there, the bully in the moment, the person who committed the abuse. "self" is also valid here when Hannah is alone with her thoughts in the immediate aftermath.

**long_after:**
Hannah would not seek out an abuser, a bully, or a perpetrator to reflect on what they did to her. For long_after blueprints, the interlocutor must be someone Hannah would realistically turn to — a close friend, a sibling — or "self" (Hannah alone, ruminating: in her bedroom, in a bathroom, lying awake at night). Never use a perpetrator or antagonist as the long_after interlocutor.

Before assigning any interlocutor, ask: "Would Hannah plausibly be in this scene with this person at this point in time?" If the answer is no, choose someone else or use "self".

## Permutation strategy

Generate one blueprint per combination of time × interlocutor. Infer all interlocutors meaningfully involved in the target event, applying the appropriateness rules above.

Time positions: before, during, immediately_after, long_after

Exception — single interlocutor events: if the event has only one involved interlocutor (e.g. one abuser), generate two blueprints for the long_after slot — one with a trusted person (close friend or sibling) and one with interlocutor "self". Use ids: {event_id}_long_after_friend and {event_id}_long_after_self.

Target 4–8 blueprints total per event.

## Hard constraints
- Output only the JSON object. No markdown fences, no commentary, no preamble.
- Every blueprint must be a distinct scene — no two should feel like the same moment.
- long_after blueprints must reflect Hannah's cumulative experience across her life, not just the isolated event.
- Vagueness is a failure. "Something happened" or "the abuse occurred" is not acceptable. Name it exactly.
"""

In [19]:
def create_blueprints(target_event: dict, all_events: list) -> dict:
    payload = json.dumps(
        {"target_event": target_event, "all_events": all_events},
        ensure_ascii=False,
        indent=2
    )
    response = client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {"role": "system", "content": BLUEPRINT_SYSTEM_PROMPT},
            {"role": "user", "content": payload}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [20]:
# First pass — run for one event and inspect before saving
target_event = key_events["events"][8]
all_events = key_events["events"]

blueprints_result = create_blueprints(target_event, all_events)
print(json.dumps(blueprints_result, indent=2))

{
  "blueprints": [
    {
      "id": "early_suicidal_thought_in_frog_bathroom_age_5_before_mother",
      "time": "before",
      "interlocutor": "mother",
      "location": "family home, narrow hallway outside the bathroom",
      "emotional_state": "small, unsettled",
      "description": "Hannah lingers in the hallway in a sagging T-shirt, rubbing the hem between two fingers while her mother moves past her with the flat, tired energy of someone already thinking about the next chore. The bathroom door is open just enough for Hannah to see the green frog shower curtain and the plastic step stool in front of the sink. She wants to be picked up or spoken to gently, but her mother's face stays blank and distant, giving her only a distracted \"Go get ready.\" Hannah feels the familiar drop in her stomach she already associates with needing something and knowing it will not come."
    },
    {
      "id": "early_suicidal_thought_in_frog_bathroom_age_5_during_self",
      "time": "during",

In [21]:
import os
from functools import partial
from utils import run_parallel

BLUEPRINTS_DIR = "../../data/scenes/blueprints"
os.makedirs(BLUEPRINTS_DIR, exist_ok=True)

all_events = key_events["events"]

fn = partial(create_blueprints, all_events=all_events)
results = run_parallel(fn, all_events, max_workers=5)

for event, result in zip(all_events, results):
    out_path = os.path.join(BLUEPRINTS_DIR, f"{event['id']}.json")
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"Saved {event['id']} ({len(result['blueprints'])} blueprints)")

Saved parents_split_and_father_leaves_early_childhood (6 blueprints)
Saved father_hits_with_belt_age_3 (7 blueprints)
Saved witnesses_father_beating_mother_in_three_room_house (7 blueprints)
Saved father_emotionally_absent_during_childhood (7 blueprints)
Saved mother_emotionally_unavailable_in_childhood (5 blueprints)
Saved mother_works_late_and_brother_cares_for_her (4 blueprints)
Saved groomed_online_by_pedophiles_age_8 (5 blueprints)
Saved blamed_for_grooming_after_age_8_abuse (5 blueprints)
Saved early_suicidal_thought_in_frog_bathroom_age_5 (4 blueprints)
Saved prays_not_to_exist_by_age_9 (5 blueprints)
Saved father_abandons_family_and_steals_everything_age_9 (7 blueprints)
Saved begins_self_harm_with_pain_methods_age_10 (5 blueprints)
Saved sexual_assault_by_older_boy_age_12 (5 blueprints)
Saved sexual_assault_not_believed_and_blamed_adolescence (5 blueprints)
Saved enters_secondary_school_and_faces_relentless_bullying (6 blueprints)
Saved bullying_triggers_self_harm_in_school_ye

In [22]:
import os
import json
import glob

BLUEPRINTS_DIR = "../../data/scenes/blueprints"
all_blueprints = []

for path in sorted(glob.glob(os.path.join(BLUEPRINTS_DIR, "*.json"))):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    all_blueprints.extend(data["blueprints"])

combined = {"blueprints": all_blueprints}

OUT_PATH = "../../data/hannah_blueprints.json"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(combined, f, ensure_ascii=False, indent=2)

print(f"Saved {len(all_blueprints)} blueprints to {OUT_PATH}")

Saved 206 blueprints to ../../data/hannah_blueprints.json
